In [2]:
import torch
import sys

def check_pytorch_cuda():
    print("Checking CUDA for PyTorch...")
    if torch.cuda.is_available():
        print(f"PyTorch CUDA is available!")
        print(f"PyTorch CUDA Version: {torch.version.cuda}")
        print(f"Number of GPUs detected by PyTorch: {torch.cuda.device_count()}")
        print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("CUDA is NOT available in PyTorch!")

def main():
    print("=== CUDA Environment Check ===")
    check_pytorch_cuda()

if __name__ == "__main__":
    main()


=== CUDA Environment Check ===
Checking CUDA for PyTorch...
PyTorch CUDA is available!
PyTorch CUDA Version: 12.4
Number of GPUs detected by PyTorch: 1
GPU Device Name: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
!pip install matplotlib


In [6]:
from PIL import Image
import PIL

# Increase the pixel limit (for large images)
PIL.Image.MAX_IMAGE_PIXELS = None  # Disable the limit entirely

# Continue with your image processing code


In [7]:
import os
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as patches

# Define paths and parameters
input_folder = r"dataset - Copy"  # Folder containing original images and labels
output_folder = r"resized_dataset"  # Folder to save resized images and labels together
os.makedirs(output_folder, exist_ok=True)

# Define new size for resizing images
new_width = 600  # target width
new_height = 800  # target height

def read_yolo_labels(label_path, img_width, img_height):
    """Read YOLO format label file and convert to pixel coordinates."""
    bboxes = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id, x_center, y_center, width, height = map(float, parts[:5])
                    # Convert YOLO format to pixel coordinates
                    x_center *= img_width
                    y_center *= img_height
                    width *= img_width
                    height *= img_height
                    x_min = x_center - width/2
                    y_min = y_center - height/2
                    x_max = x_center + width/2
                    y_max = y_center + height/2
                    bboxes.append([x_min, y_min, x_max, y_max])
    return bboxes

def draw_bounding_boxes(ax, image, bboxes, color='r'):
    """Draw bounding boxes on the given axis."""
    ax.imshow(image)
    for bbox in bboxes:
        x_min, y_min, x_max, y_max = bbox
        width = x_max - x_min
        height = y_max - y_min
        rect = patches.Rectangle((x_min, y_min), width, height, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)

def process_image_and_labels(image_name, image_path, output_folder):
    """Process a single image and its corresponding labels."""
    try:
        # Open and get original image dimensions
        image = Image.open(image_path)
        original_width, original_height = image.size
        
        # Calculate scaling factors
        scale_x = new_width / original_width
        scale_y = new_height / original_height
        
        # Resize the image
        resized_image = image.resize((new_width, new_height))
        
        # Save the resized image
        resized_image_path = os.path.join(output_folder, f"resized_{image_name}")
        resized_image.save(resized_image_path)
        
        # Check for corresponding label file
        base_name = os.path.splitext(image_name)[0]
        label_file = f"{base_name}.txt"
        label_path = os.path.join(input_folder, label_file)
        
        # Read original annotations
        original_bboxes = read_yolo_labels(label_path, original_width, original_height)
        
        if original_bboxes:
            adjusted_annotations = []
            label_content = ""
            
            for bbox in original_bboxes:
                x_min, y_min, x_max, y_max = bbox
                # Adjust bounding box coordinates
                new_bbox = [
                    int(x_min * scale_x),
                    int(y_min * scale_y),
                    int(x_max * scale_x),
                    int(y_max * scale_y)
                ]
                adjusted_annotations.append(new_bbox)
                
                # Convert back to YOLO format (normalized center coordinates and dimensions)
                norm_x_center = ((new_bbox[0] + new_bbox[2]) / 2) / new_width
                norm_y_center = ((new_bbox[1] + new_bbox[3]) / 2) / new_height
                norm_width = (new_bbox[2] - new_bbox[0]) / new_width
                norm_height = (new_bbox[3] - new_bbox[1]) / new_height
                
                # Add to label content (class 0 assumed - change if needed)
                label_content += f"0 {norm_x_center:.6f} {norm_y_center:.6f} {norm_width:.6f} {norm_height:.6f}\n"
            
            # Save the new label file
            new_label_file = os.path.join(output_folder, f"resized_{base_name}.txt")
            with open(new_label_file, 'w') as f:
                f.write(label_content)
            
            # Create comparison visualization - only for the first few images to prevent crashes
            if len(os.listdir(output_folder)) < 10:  # Only create comparisons for first 10 images
                fig, axes = plt.subplots(1, 2, figsize=(12, 6))
                axes[0].set_title("Original Image with BBox")
                draw_bounding_boxes(axes[0], image, original_bboxes, color='b')
                
                axes[1].set_title("Resized Image with Adjusted BBox")
                draw_bounding_boxes(axes[1], resized_image, adjusted_annotations, color='r')
                
                for ax in axes:
                    ax.axis('off')
                
                comparison_path = os.path.join(output_folder, f"comparison_{base_name}.png")
                plt.savefig(comparison_path, bbox_inches='tight')
                plt.close(fig)  # Explicitly close the figure to free memory
            
            print(f"Processed {image_name} with annotations")
        else:
            print(f"Processed {image_name} (no annotations found)")
    except Exception as e:
        print(f"Error processing {image_name}: {str(e)}")

# Process all images in the input folder
for image_name in sorted(os.listdir(input_folder)):
    if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        image_path = os.path.join(input_folder, image_name)
        process_image_and_labels(image_name, image_path, output_folder)

print("All images and labels processed and saved in:", output_folder)

Processed IMG_20250227_103851 - Copy.jpg with annotations
Processed IMG_20250227_103851.jpg with annotations
Processed IMG_20250227_103853 - Copy.jpg with annotations
Processed IMG_20250227_103853.jpg with annotations
Processed IMG_20250227_103857 - Copy.jpg with annotations
Processed IMG_20250227_103857.jpg with annotations
Processed IMG_20250227_103859 - Copy.jpg with annotations
Processed IMG_20250227_103859.jpg with annotations
Processed IMG_20250227_103903 - Copy.jpg with annotations
Processed IMG_20250227_103903.jpg with annotations
Processed IMG_20250227_103905 - Copy.jpg with annotations
Processed IMG_20250227_103905.jpg with annotations
Processed IMG_20250227_103907 - Copy.jpg with annotations
Processed IMG_20250227_103907.jpg with annotations
Processed IMG_20250227_103909 - Copy.jpg with annotations
Processed IMG_20250227_103909.jpg with annotations
Processed IMG_20250227_103911 - Copy.jpg with annotations
Processed IMG_20250227_103911.jpg with annotations
Processed IMG_20250

# DATA SPLIT


In [2]:
import os
import shutil
import random

# Define paths
dataset_path = r"/mnt/e/_clgproject/new_project/dataset - Copy"  # Change this to your dataset folder
output_dir = r"/mnt/e/_clgproject/new_project/Yolov11_1"  # Change this to your desired output location

# Define split ratios
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# Get all image files (assuming they're .jpg, adjust if needed)
image_files = [f for f in os.listdir(dataset_path) if f.endswith(".jpg")]
random.shuffle(image_files)  # Shuffle to ensure randomness

# Calculate split sizes
num_images = len(image_files)
train_count = int(num_images * train_ratio)
val_count = int(num_images * val_ratio)
test_count = num_images - train_count - val_count  # Remaining for test

# Split the data
train_files = image_files[:train_count]
val_files = image_files[train_count:train_count + val_count]
test_files = image_files[train_count + val_count:]

# Function to copy images and labels
def copy_files(file_list, split_name):
    img_dest = os.path.join(output_dir, "images", split_name)
    lbl_dest = os.path.join(output_dir, "labels", split_name)
    os.makedirs(img_dest, exist_ok=True)
    os.makedirs(lbl_dest, exist_ok=True)

    for file in file_list:
        img_src = os.path.join(dataset_path, file)
        lbl_src = os.path.join(dataset_path, file.replace(".jpg", ".txt"))  # Assuming YOLO format
        
        # Copy image
        shutil.copy2(img_src, os.path.join(img_dest, file))

        # Copy annotation if exists
        if os.path.exists(lbl_src):
            shutil.copy2(lbl_src, os.path.join(lbl_dest, file.replace(".jpg", ".txt")))

# Copy files to respective folders
copy_files(train_files, "train")
copy_files(val_files, "val")
copy_files(test_files, "test")

print("Dataset split and copied successfully!")

Dataset split and copied successfully!


# TRAINING

In [1]:
# training yolov11 
from ultralytics import YOLO
import torch
import sys

def check_pytorch_cuda():
    print("Checking CUDA for PyTorch...")
    if torch.cuda.is_available():
        print(f"PyTorch CUDA is available!")
        print(f"PyTorch CUDA Version: {torch.version.cuda}")
        print(f"Number of GPUs detected by PyTorch: {torch.cuda.device_count()}")
        print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("CUDA is NOT available in PyTorch!")

def main():
    print("=== CUDA Environment Check ===")
    check_pytorch_cuda()

if __name__ == "__main__":
    main()

model = YOLO(r"/mnt/e/_clgproject/new_project/yolo11n.pt")
results = model.train(data=r"/mnt/e/_clgproject/new_project/V8_final_dataset/V8_final_dataset/data.yaml",workers=2, epochs=50, imgsz=640, device=0,batch=4)
print(results)

=== CUDA Environment Check ===
Checking CUDA for PyTorch...
PyTorch CUDA is available!
PyTorch CUDA Version: 12.4
Number of GPUs detected by PyTorch: 1
GPU Device Name: NVIDIA GeForce RTX 4050 Laptop GPU
New https://pypi.org/project/ultralytics/8.3.98 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.97 🚀 Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: task=detect, mode=train, model=/mnt/e/_clgproject/new_project/yolo11n.pt, data=/mnt/e/_clgproject/new_project/V8_final_dataset/V8_final_dataset/data.yaml, epochs=50, time=None, patience=100, batch=4, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropo

train: Scanning /mnt/e/_clgproject/new_project/V8_final_dataset/V8_final_dataset/labels/train.cache... 420 images, 0 backgrounds, 0 corrupt: 100%|██████████| 420/420 [00:00<?, ?it/s]
val: Scanning /mnt/e/_clgproject/new_project/V8_final_dataset/V8_final_dataset/labels/val.cache... 120 images, 0 backgrounds, 0 corrupt: 100%|██████████| 120/120 [00:00<?, ?it/s]


Plotting labels to runs/detect/train/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000714, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to runs/detect/train
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/50     0.646G      2.115      4.998      2.208          7        640: 100%|██████████| 105/105 [04:14<00:00,  2.42s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:20<00:00,  1.36s/it]

                   all        120        120      0.957     0.0158       0.13     0.0548


...

      50/50     0.803G      1.216      0.997      1.538          4        640: 100%|██████████| 105/105 [02:35<00:00,  1.48s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:13<00:00,  1.08it/s]

                   all        120        120      0.974      0.968      0.985      0.535



50 epochs completed in 2.654 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 5.5MB
Optimizer stripped from runs/detect/train/weights/best.pt, 5.5MB

Validating runs/detect/train/weights/best.pt...
Ultralytics 8.3.97 🚀 Python-3.10.8 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11n summary (fused): 100 layers, 2,584,102 parameters, 0 gradients, 6.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:22<00:00,  1.48s/it]


                   all        120        120      0.971      0.985      0.992      0.557
                 hello         10         10      0.968          1      0.995      0.581
              ILOVEYOU         19         19      0.979          1      0.995      0.702
                   YES         12         12          1      0.941      0.995      0.524
                    NO         10         10      0.983          1      0.995      0.463
             THANK YOU         14         14      0.884      0.929      0.962      0.645
                PLEASE         13         13      0.967          1      0.995      0.544
                  I/ME          7          7      0.971          1      0.995      0.509
                  WANT         12         12      0.983          1      0.995      0.563
                    GO          9          9      0.976          1      0.995      0.574
                 WATER         14         14          1      0.982      0.995      0.465
Speed: 1.1ms preproce

In [4]:
from ultralytics import YOLO

# Load the trained YOLOv8 model
model = YOLO(r"E:\_clgproject\new_project\runs_l_good_W\detect\train2\weights\best.pt")

# Run real-time detection using webcam
model.predict(source=0, show=True)


1/1: 0... Success  (inf frames of shape 640x480 at 30.00 FPS)


WARNING  inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

0: 480x640 (no detections), 337.6ms
0: 480x640 (no detections), 273.0ms
0: 480x640 (no detections), 291.6ms
0: 480x640 (no detections), 263.7ms
0: 480x640 (no detections), 257.6ms
0: 480x640 (no detections), 264.3ms
0: 480x640 (no detections), 277.3ms
0: 480x640 (no detections), 369.7ms
0: 480x640 (no detections), 449.6ms
0: 480x640 (no detections), 398.6ms
0: 480x640 (no detections), 409.9ms

KeyboardInterrupt: 

In [6]:
from ultralytics import YOLO
import cv2

# Load model
model = YOLO(r"E:\_clgproject\new_project\runs_l_good_W\detect\train2\weights\best.pt")

# Custom webcam loop for more control
cap = cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Process every other frame if needed (skip n frames)
    results = model(frame, verbose=False)  # verbose=False reduces console output
    
    # Display results
    annotated_frame = results[0].plot()
    cv2.imshow('YOLOv8 Detection', annotated_frame)
    
    if cv2.waitKey(0) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [7]:
from ultralytics import YOLO
import cv2

# Load YOLO model
model = YOLO(r"E:\_clgproject\new_project\runs_l_good_W\detect\train2\weights\best.pt")

# Initialize video capture
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)

    # Process detections and print words in real-time
    detected_words = set()  # Store detected words in the current frame

    for box in results[0].boxes:
        class_index = int(box.cls)
        word = model.names[class_index]  # Convert index to word using YOLO class names
        detected_words.add(word)

    # Print each detected word without cooldown
    for word in detected_words:
        print(f"Detected Word: {word}")

    # Display annotated frame
    annotated_frame = results[0].plot()
    cv2.imshow('Word Detection', annotated_frame)

    # Exit condition on 'q' key
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Detected Word: GO
Detected Word: GO


KeyboardInterrupt: 

In [10]:
from ultralytics import YOLO
import cv2
from collections import Counter

# Load YOLO model
model = YOLO(r"E:\_clgproject\new_project\runs_l_good_W\detect\train2\weights\best.pt")

# Initialize video capture
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# Initialize word tracking and cooldown
word_counter = Counter()
cooldown = 0

def filter_and_generate_sentence(counter):
    """Generate a sentence from detected word frequencies."""
    words = [word for word, count in counter.items() if count > 1]  # Filter stable words
    return " ".join(words).capitalize() if words else None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    current_frame_words = []

    # Detect words in the current frame
    for box in results[0].boxes:
        class_index = int(box.cls)
        word = model.names[class_index].lower()
        current_frame_words.append(word)

    # Update word counter and clear words with low counts
    word_counter.update(current_frame_words)

    # Generate and print a sentence if cooldown allows
    if cooldown <= 0:
        sentence = filter_and_generate_sentence(word_counter)
        if sentence:
            print(f"Generated Sentence: {sentence}")
            word_counter.clear()  # Clear after sentence generation
            cooldown = 0  # Frame cooldown to prevent immediate sentence regeneration

    # Display the annotated frame
    annotated_frame = results[0].plot()
    cv2.imshow('Word Detection and Sentence Formation', annotated_frame)

    # Exit condition on 'q' key
    if cv2.waitKey(1) == ord('q'):
        break

    cooldown -= 1

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

In [1]:
from ultralytics import YOLO
import cv2
from collections import Counter

# Load YOLO model
model = YOLO(r"E:\_clgproject\new_project\yolov11m_best_10.pt")

# Initialize video capture
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# Initialize word tracking and cooldown
word_counter = Counter()
no_action_frames = 0
max_no_action_frames =10  # Reset sentence after 15 frames of no hand detection

def filter_and_generate_sentence(counter):
    """Generate a sentence from detected word frequencies."""
    words = [word for word, count in counter.items() if count > 1]  # Filter stable words
    return " ".join(words).capitalize() if words else None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)
    current_frame_words = []

    # Detect words in the current frame
    for box in results[0].boxes:
        class_index = int(box.cls)
        word = model.names[class_index].lower()
        current_frame_words.append(word)

    # Check if hand actions are detected or not
    if current_frame_words:
        no_action_frames = 0  # Reset inactivity counter if hand action is detected
        word_counter.update(current_frame_words)
    else:
        no_action_frames += 1  # Increment counter when no hand action is detected

    # Generate and print a sentence after stable word detection or on inactivity reset
    if no_action_frames >= max_no_action_frames or (len(word_counter) > 0 and len(current_frame_words) == 0):
        sentence = filter_and_generate_sentence(word_counter)
        if sentence:
            print(f"Generated Sentence: {sentence}")
            word_counter.clear()  # Clear word count to start the next sentence
            no_action_frames = 0  # Reset inactivity counter after sentence generation

    # Display the annotated frame
    annotated_frame = results[0].plot()
    cv2.imshow('Word Detection and Sentence Formation', annotated_frame)

    # Exit condition on 'q' key
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Generated Sentence: Hello
Generated Sentence: Hello
Generated Sentence: Iloveyou
Generated Sentence: Want please water
Generated Sentence: Want
Generated Sentence: Want
Generated Sentence: Want
Generated Sentence: Want please
Generated Sentence: Want
Generated Sentence: Want
Generated Sentence: Want
Generated Sentence: No
Generated Sentence: Want
Generated Sentence: Want
